# 08 — Backpropagation, Training Loops, and Optimizers

This notebook trains an MLP to learn:

\[
(A + B) \bmod 5
\]

for all ordered pairs \(A,B \in \{0,\dots,99\}\).

The experiment connects the full learning pipeline:

```text
forward pass
    ↓
cross-entropy loss
    ↓
backpropagation / autograd
    ↓
AdamW optimizer
    ↓
updated parameters
```

It also studies how **learning rate**, **batch size**, and **model capacity**
affect convergence.

## 0 — Setup

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
print("PyTorch version:", torch.__version__)

## 1 — Build the `(A + B) % 5` Dataset

There are \(100 	imes 100 = 10{,}000\) ordered pairs and five perfectly
balanced target classes.

In [ ]:
dataset = []

for a in range(100):
    for b in range(100):
        result = (a + b) % 5
        dataset.append((a, b, result))

from collections import Counter

counts = Counter(r for _, _, r in dataset)

print("Dataset size:", len(dataset))
print("Class distribution:", dict(sorted(counts.items())))

## 2 — One-Hot Input Representation

Each example becomes a 200-dimensional vector:

- dimensions `0..99` represent \(A\)
- dimensions `100..199` represent \(B\)

In [ ]:
class Mod5Dataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        a, b, result = self.data[idx]

        x = torch.zeros(200)
        x[a] = 1.0
        x[100 + b] = 1.0

        return x, result

## 3 — MLP Architecture

Default architecture:

```text
200 → 32 → ReLU → 16 → ReLU → 5
```

The default model contains **7,045 trainable parameters**.

In [ ]:
class Mod5MLP(nn.Module):
    def __init__(self, hidden1=32, hidden2=16):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(200, hidden1),
            nn.ReLU(),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, 5),
        )

    def forward(self, x):
        return self.net(x)


model = Mod5MLP()

print(model)
print("Parameter count:", sum(p.numel() for p in model.parameters()))

## 4 — Train / Validation Split

In [ ]:
full_dataset = Mod5Dataset(dataset)

indices = list(range(len(full_dataset)))

train_indices, validation_indices = train_test_split(
    indices,
    test_size=0.3,
    random_state=42,
    shuffle=True
)

train_dataset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, validation_indices)

print(f"Train:      {len(train_dataset):,}")
print(f"Validation: {len(val_dataset):,}")

## 5 — Training Loop

The core update step is:

```text
optimizer.zero_grad()
        ↓
forward pass
        ↓
CrossEntropyLoss
        ↓
loss.backward()
        ↓
optimizer.step()
```

In [ ]:
def calculate_accuracy(model, dataloader):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in dataloader:
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total


def train_model(
    model,
    train_ds,
    val_ds,
    lr=1e-3,
    batch_size=32,
    max_epochs=50,
    verbose=True
):
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr
    )

    criterion = nn.CrossEntropyLoss()

    history = {
        "epoch": [],
        "loss": [],
        "train_acc": [],
        "val_acc": []
    }

    for epoch in range(1, max_epochs + 1):
        model.train()
        running_loss = 0.0

        for x, y in train_loader:
            optimizer.zero_grad()

            logits = model(x)
            loss = criterion(logits, y)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        train_acc = calculate_accuracy(model, train_loader)
        val_acc = calculate_accuracy(model, val_loader)

        history["epoch"].append(epoch)
        history["loss"].append(avg_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if verbose:
            print(
                f"Epoch {epoch:3d} | "
                f"Loss {avg_loss:.4f} | "
                f"Train {train_acc:.1%} | "
                f"Val {val_acc:.1%}"
            )

        if val_acc == 1.0:
            if verbose:
                print(f"\nPerfect validation accuracy at epoch {epoch}.")
            break

    return history

## 6 — Baseline Training Run

Configuration used in the recorded experiment:

```text
Optimizer:  AdamW
Loss:       CrossEntropyLoss
LR:         1e-3
Batch size: 32
Architecture: 200 → 32 → 16 → 5
```

In [ ]:
# Re-run this cell to reproduce the baseline experiment.
baseline_model = Mod5MLP()

baseline_history = train_model(
    baseline_model,
    train_dataset,
    val_dataset,
    lr=1e-3,
    batch_size=32
)

### Recorded baseline result

In the original run, validation accuracy progressed to **100% at epoch 6**:

```text
Epoch 1 | Loss 1.6118 | Train 20.2%  | Val 19.6%
Epoch 2 | Loss 1.6090 | Train 26.9%  | Val 20.4%
Epoch 3 | Loss 1.5923 | Train 50.0%  | Val 40.8%
Epoch 4 | Loss 1.3009 | Train 88.4%  | Val 85.3%
Epoch 5 | Loss 0.5677 | Train 100.0% | Val 99.9%
Epoch 6 | Loss 0.1767 | Train 100.0% | Val 100.0%
```

## 7 — Visualize Training History

In [ ]:
def plot_training_history(history):
    epochs = history["epoch"]

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["loss"], marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Cross-Entropy Loss")
    plt.title("Training Loss")
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_acc"], marker="o", label="Train")
    plt.plot(epochs, history["val_acc"], marker="o", label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training vs Validation Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()


# Uncomment after running the baseline training cell:
# plot_training_history(baseline_history)

## 8 — Helper: Epochs to Convergence

A fresh model is trained for every hyperparameter setting.

Convergence is defined as reaching **100% validation accuracy** within the
experiment's epoch budget.

In [ ]:
def epochs_to_converge(
    lr,
    batch_size=32,
    max_epochs=25,
    hidden1=32,
    hidden2=16
):
    model = Mod5MLP(
        hidden1=hidden1,
        hidden2=hidden2
    )

    history = train_model(
        model,
        train_dataset,
        val_dataset,
        lr=lr,
        batch_size=batch_size,
        max_epochs=max_epochs,
        verbose=False
    )

    if history["val_acc"][-1] == 1.0:
        return history["epoch"][-1]

    return max_epochs

## 9 — Learning-Rate Sweep

The original experiment tested learning rates from `1e-5` to `3e-1`.

Very small learning rates learned too slowly within 25 epochs, while very
large values became ineffective. Intermediate values converged much faster.

In [ ]:
learning_rates = [
    1e-5, 5e-5, 1e-4, 5e-4, 1e-3,
    3e-3, 1e-2, 3e-2, 1e-1, 3e-1
]

# Full rerun (computationally expensive):
#
# lr_results = {}
# for lr in learning_rates:
#     lr_results[lr] = epochs_to_converge(lr)
#     print(lr, lr_results[lr])

# Recorded results from the completed experiment:
lr_results = {
    1e-5: 25,
    5e-5: 25,
    1e-4: 25,
    5e-4: 12,
    1e-3: 5,
    3e-3: 3,
    1e-2: 3,
    3e-2: 13,
    1e-1: 25,
    3e-1: 25,
}

print(lr_results)

In [ ]:
lrs = list(lr_results.keys())
epochs = list(lr_results.values())

plt.figure(figsize=(9, 5))
plt.plot(lrs, epochs, marker="o")
plt.xscale("log")
plt.xlabel("Learning Rate")
plt.ylabel("Epochs to 100% Validation Accuracy")
plt.title("Learning Rate vs Convergence Speed")
plt.grid(True)
plt.show()

### Learning-rate observation

The fastest recorded convergence occurred at both `3e-3` and `1e-2`
(3 epochs). The later workshop experiments continued with `1e-3` as a
stable reference value rather than claiming it was the numerically fastest
learning rate in this particular random run.

## 10 — Batch-Size Sweep

In [ ]:
batch_sizes = [
    4, 8, 16, 32, 64,
    128, 256, 512, 1024, 2000
]

# Full rerun:
#
# bs_results = {}
# for bs in batch_sizes:
#     bs_results[bs] = epochs_to_converge(
#         lr=1e-3,
#         batch_size=bs
#     )

# Recorded results:
bs_results = {
    4: 3,
    8: 4,
    16: 4,
    32: 6,
    64: 9,
    128: 12,
    256: 18,
    512: 25,
    1024: 25,
    2000: 25,
}

print(bs_results)

In [ ]:
bss = list(bs_results.keys())
epochs = list(bs_results.values())

plt.figure(figsize=(9, 5))
plt.plot(bss, epochs, marker="s")
plt.xscale("log")
plt.xlabel("Batch Size")
plt.ylabel("Epochs to 100% Validation Accuracy")
plt.title("Batch Size vs Convergence Speed")
plt.grid(True)
plt.show()

### Batch-size observation

In the recorded run:

- batch size `4` reached 100% validation accuracy in 3 epochs
- `8` and `16` required 4 epochs
- larger batches converged progressively more slowly
- `512`, `1024`, and `2000` did not reach perfect validation accuracy
  within the 25-epoch budget

This experiment measures **epochs**, not total optimizer steps or wall-clock
time, so the result should be interpreted in that context.

## 11 — Model Capacity Sweep

In [ ]:
hidden_configs = [
    (4, 2),
    (8, 4),
    (16, 8),
    (32, 16),
    (64, 32),
    (128, 64),
    (256, 128),
    (512, 256),
    (1024, 512),
    (2048, 1024),
]

# Recorded results from lr=1e-3, batch_size=32:
capacity_results = [
    # (hidden1, hidden2, parameter_count, epochs)
    (4, 2, 829, 25),
    (8, 4, 1669, 25),
    (16, 8, 3397, 9),
    (32, 16, 7045, 6),
    (64, 32, 15109, 4),
    (128, 64, 34309, 4),
    (256, 128, 84997, 3),
    (512, 256, 235525, 3),
    (1024, 512, 733189, 3),
    (2048, 1024, 2514949, 3),
]

for h1, h2, params, ep in capacity_results:
    status = f"{ep} epochs" if ep < 25 else "did not converge"
    print(f"{h1:4d}/{h2:<4d} | {params:>8,d} params | {status}")

In [ ]:
params = [row[2] for row in capacity_results]
epochs = [row[3] for row in capacity_results]

plt.figure(figsize=(9, 5))
plt.plot(params, epochs, marker="D")
plt.xscale("log")
plt.xlabel("Total Parameters")
plt.ylabel("Epochs to 100% Validation Accuracy")
plt.title("Model Capacity vs Convergence Speed")
plt.grid(True)
plt.show()

### Capacity observation

The smallest two models did not converge within 25 epochs.

Increasing model capacity reduced the number of epochs required until the
experiment reached a plateau around 3 epochs. This is evidence of
diminishing returns for this particular task and metric.

## 12 — Minimal Training Loop

The completed exercise used:

```text
learning rate = 1e-3
batch size    = 4
optimizer     = AdamW
loss          = CrossEntropyLoss
```

In [ ]:
best_lr = 1e-3
best_batch_size = 4
epochs = 25

model = Mod5MLP()

train_loader = DataLoader(
    train_dataset,
    batch_size=best_batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=best_batch_size
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=best_lr
)

criterion = nn.CrossEntropyLoss()

for epoch in range(1, epochs + 1):
    model.train()
    epoch_loss = 0.0

    for x, y in train_loader:
        optimizer.zero_grad()

        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    val_acc = calculate_accuracy(
        model,
        val_loader
    )

    print(
        f"Epoch {epoch} | "
        f"Loss: {epoch_loss / len(train_loader):.4f} | "
        f"Val Acc: {val_acc:.1%}"
    )

    if val_acc == 1.0:
        print(
            f"\nPerfect validation accuracy "
            f"reached at epoch {epoch}."
        )
        break

## Key Observations

- `loss.backward()` computes gradients through PyTorch autograd.
- `optimizer.step()` updates model parameters; backward propagation alone
  does not change the weights.
- `optimizer.zero_grad()` is required because PyTorch gradients accumulate.
- Learning rate strongly affects convergence: too small can be extremely
  slow, while too large can make optimization ineffective.
- Batch size changes both gradient behavior and the number of optimizer
  updates per epoch.
- A model can be too small to learn the task within a fixed training budget.
- More capacity initially improved convergence, but the benefit eventually
  plateaued.
- Train/validation separation makes it possible to evaluate learned
  behavior on held-out examples.

# Connection to the LLM Roadmap

The same conceptual training loop scales to language models:

```text
token batch
    ↓
transformer
    ↓
vocabulary logits
    ↓
cross entropy
    ↓
backpropagation
    ↓
optimizer
    ↓
updated weights
```

What changes at LLM scale is the size of the model, dataset, hardware, and
optimization system—not the basic learning logic demonstrated here.